##  重试解析器

虽然在某些情况下，可以仅通过查看输出来修复任何解析错误，但在其他情况下则不然。例如，输出不仅格式不正确，而且部分完整。考虑下面的例子。

In [32]:
import pprint
from langchain.output_parsers import (
    OutputFixingParser,
    PydanticOutputParser,
)
from langchain.prompts import (
    PromptTemplate,
)
# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel
from langchain_openai import ChatOpenAI, OpenAI
openai_api_key = "EMPTY"
openai_api_base = "http://127.0.0.1:1234/v1"
chat = ChatOpenAI(
    openai_api_key=openai_api_key,
    openai_api_base=openai_api_base,
    temperature=0.3,
)

In [33]:
template = """根据用户问题，提供应采取的操作和操作输入。
{format_instructions}
问题: {query}
答复:"""


class Action(BaseModel):
    action: str = Field(description="采取的行动")
    action_input: str = Field(description="输入到动作")


parser = PydanticOutputParser(pydantic_object=Action)

In [34]:
format_instructions = parser.get_format_instructions()
format_instructions

/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value default=PydanticUndefined description='采取的行动' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/home/vincent/anaconda3/envs/glm_py310_env/lib/python3.10/site-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value default=PydanticUndefined description='输入到动作' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"action": {"title": "Action", "type": "string"}, "action_input": {"title": "Action Input", "type": "string"}}}\n```'

In [35]:
prompt = PromptTemplate(
    template="回答用户的询问。\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [36]:
prompt_value = prompt.format_prompt(query="who is leo di caprios gf?")

In [37]:
testLLM = prompt | chat 
testLLM.invoke({"query": "who is leo di caprios gf?"})

AIMessage(content='<think>\n好的，用户问的是“who is leo di caprios gf?”，我需要回答关于莱奥·迪卡普里奥的高尔夫球手身份的问题。首先，我得确认用户指的是谁。莱奥·迪卡普里奥（Leo DiCaprio）是著名的演员，但并不是知名的高尔夫球手。不过，可能用户有拼写错误或者误解了名字。\n\n接下来，我需要确定用户是否在问某个特定人物的高尔夫背景。比如，是否有名人以“G.F.”作为绰号或昵称？例如，有些名人可能会用缩写或别名，但通常不会是“G.F.”这样的组合。不过，也有可能用户指的是某位名人，比如可能有其他同名的人，但可能性较低。\n\n然后，我需要检查是否有关于莱奥·迪卡普里奥的高尔夫活动的信息。根据我的知识，莱奥·迪卡普里奥作为演员，确实参与过一些高尔夫相关的活动，但并不是以高尔夫球手的身份闻名。他可能在某些活动中参加高尔夫比赛，但并非主要职业。\n\n因此，用户的问题可能是基于错误的信息或误解。正确的回答应该是指出莱奥·迪卡普里奥不是以高尔夫球手的身份知名，并且可能涉及拼写错误或其他混淆。\n\n最后，按照要求的JSON格式来组织答案，确保符合schema的要求，即包含“action”和“action_input”两个字段，其中“action”是动作描述，而“action_input”是输入内容。需要将问题转化为正确的JSON结构。\n</think>\n\n{\n  "action": "Answer the question about Leo DiCaprio\'s golfing career",\n  "action_input": "who is Leo DiCaprio\'s GF?"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 358, 'prompt_tokens': 175, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'qwen/qwen3-1.7b', 'system_finger

In [38]:
chain = prompt | chat | parser
parser_output = chain.invoke({"query": "who is leo di caprios gf?"})

parser_output

OutputParserException: Invalid json output: <think>
Okay, let me try to figure out the answer. The user is asking about Leo Di Caprio and his relationship with GF. First, I need to recall who Leo Di Caprio is. He's a well-known actor, right? Known for movies like "Titanic" and "The Departed." But wait, he's also a philanthropist and activist. There's a charity called the Leo DiCaprio Foundation, which focuses on environmental sustainability and social justice.

Now, the user might be asking about his personal life, specifically his relationship with GF. Wait, GF stands for "Gulf and Fox," but I think that's a company name. However, in the context of the question, maybe the user is referring to someone else? Or perhaps there's a mix-up here. Let me check.

Wait, the user might be asking about Leo Di Caprio's girlfriend or fiancé. But I need to confirm if there's any public information about his personal relationships. However, as an AI, I should not make up information. The user might b

In [39]:
bad_response = '{"action": "search"}'

In [40]:
parser.parse(bad_response)

Action(action='search', action_input=FieldInfo(default=PydanticUndefined, description='输入到动作', extra={}))

In [41]:
fix_parser = OutputFixingParser.from_llm(parser=parser, llm=chat)

In [42]:
fix_parser.parse(bad_response)

Action(action='search', action_input=FieldInfo(default=PydanticUndefined, description='输入到动作', extra={}))

In [43]:
from langchain.output_parsers import RetryOutputParser
retry_parser = RetryOutputParser.from_llm(parser=parser, llm=chat)



In [44]:
retry_parser.parse_with_prompt(bad_response, prompt_value)

Action(action='search', action_input=FieldInfo(default=PydanticUndefined, description='输入到动作', extra={}))